In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/digit-recognizer/sample_submission.csv
/kaggle/input/competitions/digit-recognizer/train.csv
/kaggle/input/competitions/digit-recognizer/test.csv


In [2]:
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

# Fewer epochs on CPU so it finishes in reasonable time
EPOCHS = 25 if device == "cuda" else 8
BATCH = 128 if device == "cuda" else 256
LR = 2e-3

Using: cuda


In [3]:
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found. Add the Digit Recognizer competition as input."
    return hits[0]

try:
    train_df = pd.read_csv(find("train.csv"), engine="pyarrow")
    test_df = pd.read_csv(find("test.csv"), engine="pyarrow")
except Exception:
    train_df = pd.read_csv(find("train.csv"))
    test_df = pd.read_csv(find("test.csv"))

print(train_df.shape, test_df.shape)  # expect (42000, 785) (28000, 784)

y = train_df["label"].values
X = train_df.drop(columns="label").values.astype(np.float32) / 255.0
X = X.reshape(-1, 1, 28, 28)
X_test = (test_df.values.astype(np.float32) / 255.0).reshape(-1, 1, 28, 28)

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.1, random_state=SEED, stratify=y
)
print(X_tr.shape, X_val.shape, X_test.shape)

(42000, 785) (28000, 784)
(37800, 1, 28, 28) (4200, 1, 28, 28) (28000, 1, 28, 28)


In [4]:
aug = T.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.9, 1.1))

class DigitDS(Dataset):
    def __init__(self, X, y=None, augment=False):
        self.X = torch.tensor(X)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = self.X[i]
        if self.augment:
            x = aug(x)
        x = (x - 0.1307) / 0.3081
        return x if self.y is None else (x, self.y[i])

train_dl = DataLoader(DigitDS(X_tr, y_tr, augment=True), batch_size=BATCH,
                      shuffle=True, num_workers=2)
val_dl = DataLoader(DigitDS(X_val, y_val), batch_size=256)
test_dl = DataLoader(DigitDS(X_test), batch_size=256)


In [5]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        def block(i, o):
            return nn.Sequential(
                nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(2), nn.Dropout(0.25),
            )
        self.features = nn.Sequential(block(1, 32), block(32, 64), block(64, 128))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 3 * 3, 256), nn.ReLU(),
            nn.Dropout(0.4), nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.head(self.features(x))

model = Net().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_dl)
)

In [6]:
def evaluate(dl):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += len(yb)
    return correct / total

best = 0
for ep in range(EPOCHS):
    model.train()
    running = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = F.cross_entropy(model(xb), yb, label_smoothing=0.05)
        loss.backward()
        opt.step()
        sched.step()
        running += loss.item() * len(yb)
    acc = evaluate(val_dl)
    print(f"epoch {ep+1:02d}/{EPOCHS}  loss {running/len(train_dl.dataset):.4f}  val_acc {acc:.4f}")
    if acc > best:
        best = acc
        torch.save(model.state_dict(), "best.pt")

print("Best val acc:", best)


epoch 01/25  loss 1.2399  val_acc 0.9645
epoch 02/25  loss 0.4996  val_acc 0.9793
epoch 03/25  loss 0.4192  val_acc 0.9819
epoch 04/25  loss 0.3923  val_acc 0.9917
epoch 05/25  loss 0.3714  val_acc 0.9902
epoch 06/25  loss 0.3640  val_acc 0.9907
epoch 07/25  loss 0.3563  val_acc 0.9883
epoch 08/25  loss 0.3511  val_acc 0.9926
epoch 09/25  loss 0.3443  val_acc 0.9948
epoch 10/25  loss 0.3391  val_acc 0.9933
epoch 11/25  loss 0.3367  val_acc 0.9919
epoch 12/25  loss 0.3352  val_acc 0.9929
epoch 13/25  loss 0.3315  val_acc 0.9933
epoch 14/25  loss 0.3284  val_acc 0.9952
epoch 15/25  loss 0.3260  val_acc 0.9948
epoch 16/25  loss 0.3231  val_acc 0.9960
epoch 17/25  loss 0.3199  val_acc 0.9957
epoch 18/25  loss 0.3207  val_acc 0.9964
epoch 19/25  loss 0.3164  val_acc 0.9960
epoch 20/25  loss 0.3133  val_acc 0.9957
epoch 21/25  loss 0.3128  val_acc 0.9967
epoch 22/25  loss 0.3107  val_acc 0.9967
epoch 23/25  loss 0.3094  val_acc 0.9967
epoch 24/25  loss 0.3085  val_acc 0.9969
epoch 25/25  los

In [7]:
model.load_state_dict(torch.load("best.pt", map_location=device))
model.eval()
preds = []
with torch.no_grad():
    for xb in test_dl:
        preds.append(model(xb.to(device)).argmax(1).cpu())
preds = torch.cat(preds).numpy()

sub = pd.DataFrame({"ImageId": np.arange(1, len(preds) + 1), "Label": preds})
sub.to_csv("submission.csv", index=False)
print(sub.head())


   ImageId  Label
0        1      2
1        2      0
2        3      9
3        4      0
4        5      3
